# Data Pipeline — Sanity Check

Visual verification that the data pipeline (`data/preprocessing/data_pipeline.py`) is
working correctly. Plots three spectrograms — training (SpecAugment masks visible),
validation (no augmentation), and a multi-label soundscape sample — to confirm:

1. Spectrogram shape is `(1, 128, 313)` — correct mel params
2. SpecAugment is applied to training samples only
3. Multi-label soundscape samples load and encode correctly

**Note:** This notebook imports directly from `data_pipeline.py`. It does not
duplicate pipeline code. Set `$DATA_ROOT` to your data directory before running.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# Allow import from repo root regardless of working directory
_repo_root = Path().resolve()
if (_repo_root / "data" / "preprocessing" / "data_pipeline.py").exists():
    pass  # already at repo root
else:
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from data.preprocessing.data_pipeline import build_dataloaders  # noqa: E402

DATA_ROOT = Path(
    os.environ.get("DATA_ROOT")
    or os.environ.get("BIRDCLEF_DATA_ROOT")
    or "/home/renku/work/kaggle-data/birdclef-2026"
)
assert DATA_ROOT.exists(), f"DATA_ROOT not found: {DATA_ROOT}. Set $DATA_ROOT."

train_loader, val_loader, _test_loader, mlb = build_dataloaders(
    metadata_csv=str(DATA_ROOT / "train.csv"),
    audio_dir=str(DATA_ROOT / "train_audio"),
    soundscapes_dir=str(DATA_ROOT / "train_soundscapes"),
    soundscapes_csv=str(DATA_ROOT / "train_soundscapes_labels.csv"),
    val_size=0.15,
    test_size=0.15,
    batch_size=8,
    num_workers=2,
)

fig, axes = plt.subplots(1, 3, figsize=(22, 4))

# ── Check 1: Training spectrogram (SpecAugment masks expected) ──────────────
train_specs, train_labels = next(iter(train_loader))
print(f"Train spec shape  : {train_specs.shape}")   # (8, 1, 128, 313)
print(f"Label vector shape: {train_labels.shape}")  # (8, K)
print(f"Label sums        : {train_labels.sum(dim=1).tolist()}")

ax = axes[0]
ax.imshow(train_specs[0, 0].numpy(), aspect="auto", origin="lower", cmap="viridis")
active = mlb.classes_[train_labels[0].bool().numpy()]
ax.set_title(f"[Train] {', '.join(active)}
(SpecAugment masks expected)", fontsize=9)
ax.set_xlabel("Time frames")
ax.set_ylabel("Mel bins")

# ── Check 2: Validation spectrogram (NO SpecAugment) ────────────────────────
val_specs, val_labels = next(iter(val_loader))
print(f"
Val spec shape    : {val_specs.shape}")
print(f"Label sums (all 1): {val_labels.sum(dim=1).tolist()}")

ax = axes[1]
ax.imshow(val_specs[0, 0].numpy(), aspect="auto", origin="lower", cmap="viridis")
active_val = mlb.classes_[val_labels[0].bool().numpy()]
ax.set_title(f"[Val] {', '.join(active_val)}
(no SpecAugment expected)", fontsize=9)
ax.set_xlabel("Time frames")
ax.set_ylabel("Mel bins")

# ── Check 3: Multi-label soundscape sample ───────────────────────────────────
multi_label_idx = next(
    (i for i, s in enumerate(train_loader.dataset.samples) if len(s["labels"]) > 1),
    None,
)

if multi_label_idx is not None:
    spec_ml, label_ml = train_loader.dataset[multi_label_idx]
    sample_ml = train_loader.dataset.samples[multi_label_idx]
    active_ml = mlb.classes_[label_ml.bool().numpy()]
    print(f"
Soundscape idx    : {multi_label_idx}")
    print(f"Raw labels        : {sample_ml['labels']}")
    print(f"Encoded classes   : {active_ml.tolist()}")
    print(f"Label sum         : {int(label_ml.sum())}  (expected > 1)")

    ax = axes[2]
    ax.imshow(spec_ml[0].numpy(), aspect="auto", origin="lower", cmap="viridis")
    ax.set_title(
        f"[Soundscape] {', '.join(active_ml)}
label sum={int(label_ml.sum())} (multi-label expected)",
        fontsize=9,
    )
    ax.set_xlabel("Time frames")
    ax.set_ylabel("Mel bins")
else:
    print("No multi-label soundscape sample found — check soundscapes CSV.")
    axes[2].set_title("[Soundscape] — no multi-label sample found")
    axes[2].axis("off")

plt.tight_layout()
plt.show()